In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

In [2]:
load_dotenv()

True

#### Chat Model & Template

In [3]:
chat = ChatGoogleGenerativeAI(
    model = 'gemini-3.1-flash-lite-preview',
    temperature = 0,
    max_output_tokens = 100,
    seed = 0
)

In [4]:
chat_template_skills = ChatPromptTemplate.from_template('''
Give the 5 of most important tools for a {profession}.
Answer by only listing the names.
''')

chat_template_projects = ChatPromptTemplate.from_template('''
Can you suggest 3 'must-do' intermediate level projects for a {profession}.
Answer by only listing the names.
''')

#### Parallel Chains

In [5]:
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [6]:
str_parser = StrOutputParser()

In [7]:
chain_skills = chat_template_skills | chat | str_parser

chain_projects = chat_template_projects | chat | str_parser

In [8]:
chain_parallel = RunnableParallel({'skills': chain_skills, 'projects': chain_projects}) # key can be anything

In [9]:
%%time
chain_skills.invoke({'profession': 'AI Engineer'})

CPU times: total: 62.5 ms
Wall time: 1.76 s


'1. Python\n2. PyTorch\n3. TensorFlow\n4. Hugging Face Transformers\n5. Docker'

In [10]:
%%time
chain_projects.invoke({'profession': 'AI Engineer'})

CPU times: total: 46.9 ms
Wall time: 1.94 s


'1. RAG-based Question Answering System\n2. Fine-tuned LLM for Domain-Specific Tasks\n3. End-to-End MLOps Pipeline with Model Monitoring'

In [11]:
%%time
chain_parallel.invoke({'profession': 'AI Engineer'})

CPU times: total: 15.6 ms
Wall time: 1.54 s


{'skills': '1. Python\n2. PyTorch\n3. TensorFlow\n4. Hugging Face Transformers\n5. Docker',
 'projects': '1. RAG-based Question Answering System\n2. Fine-tuned LLM for Domain-Specific Tasks\n3. End-to-End MLOps Pipeline with Model Monitoring'}

> invoking runnables in parrallel is more time efficient

> its because the sum of wall times of individual invokes is more than that of parallel invoke

#### Visualize Chain

In [12]:
chain_parallel.get_graph().print_ascii()

                  +--------------------------------+                     
                  | Parallel<skills,projects>Input |                     
                  +--------------------------------+                     
                       ****                  ****                        
                   ****                          ****                    
                 **                                  **                  
  +--------------------+                       +--------------------+    
  | ChatPromptTemplate |                       | ChatPromptTemplate |    
  +--------------------+                       +--------------------+    
             *                                            *              
             *                                            *              
             *                                            *              
+------------------------+                   +------------------------+  
| ChatGoogleGenerativeAI |            

> batch() invokes the same runnable with different input values

> RunnableParallel invokes several runnables with same input values